In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zain280/titanic-data-set")

print("Path to dataset files:", path)

100%|██████████| 22.0k/22.0k [00:00<00:00, 27.3MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/zain280/titanic-data-set/versions/1


In [ ]:
import os
import pandas as pd

# List files in the downloaded path to find the CSV file
files = os.listdir(path)
print("Files in dataset directory:", files)

Files in dataset directory: ['train.csv']


Assuming the main dataset file is named `Titanic-Dataset.csv` (or similar), we can load it into a pandas DataFrame. If the file name is different, you might need to adjust the code below. I will use a common name as an example, but if that throws an error, you will need to look at the list of files to get the correct name.

In [ ]:
# Load the 'train.csv' file into a pandas DataFrame
df = pd.read_csv(os.path.join(path, 'train.csv'))

print("\nFirst 5 rows of the dataset:")
display(df.head())


First 5 rows of the dataset:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# Get a concise summary of the DataFrame, including data types and non-null values
print("\nDataFrame Info:")
df.info()


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [ ]:
# Get descriptive statistics for numerical columns
print("\nDescriptive Statistics for Numerical Columns:")
display(df.describe())


Descriptive Statistics for Numerical Columns:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:
# Get descriptive statistics for categorical columns (including object type)
print("\nDescriptive Statistics for Categorical Columns:")
display(df.describe(include=['object']))


Descriptive Statistics for Categorical Columns:


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Dooley, Mr. Patrick",male,347082,G6,S
freq,1,577,7,4,644


In [ ]:
# Calculate the number of missing values per column
missing_values_count = df.isnull().sum()

# Calculate the percentage of missing values per column
missing_values_percentage = (df.isnull().sum() / len(df)) * 100

# Create a DataFrame to display missing values information
missing_info = pd.DataFrame({
    'Missing Count': missing_values_count,
    'Missing Percentage': missing_values_percentage
})

# Filter to show only columns with missing values and sort by percentage
missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("\nMissing Values Information:")
display(missing_info)


Missing Values Information:


,Missing Count,Missing Percentage
Cabin,687,77.104377
Age,177,19.865320
Embarked,2,0.224467


From the missing values information, we can see that:

*   **Cabin** has a very high percentage of missing values (over 77%). Imputing this might not be meaningful, so we might consider dropping this column or creating a new feature indicating if a `Cabin` value exists.
*   **Age** has about 19.8% missing values. This is a significant numerical feature, so imputation (e.g., with the mean, median, or a more sophisticated method) would be a good approach.
*   **Embarked** has only two missing values (0.22%). For such a small number, we can either drop these rows or impute them with the most frequent value (mode).

### Handling Missing Values

Based on the analysis, we'll proceed as follows:

1.  **Cabin**: Drop this column due to a very high percentage of missing values (over 77%), making imputation unreliable.
2.  **Age**: Impute missing values with the median age. The median is a robust measure against outliers, which can be present in age distributions.
3.  **Embarked**: Impute the two missing values with the most frequent port of embarkation (mode).

In [ ]:
# Drop the 'Cabin' column
df = df.drop(columns=['Cabin'])
print("Dropped 'Cabin' column.")

Dropped 'Cabin' column.


In [ ]:
# Impute 'Age' with the median
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)
print(f"Imputed 'Age' with median value: {median_age}.")

Imputed 'Age' with median value: 28.0.


/tmp/ipykernel_7173/3348923513.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(median_age, inplace=True)


In [ ]:
# Impute 'Embarked' with the mode
mode_embarked = df['Embarked'].mode()[0] # .mode() returns a Series, so we take the first element
df['Embarked'].fillna(mode_embarked, inplace=True)
print(f"Imputed 'Embarked' with mode value: {mode_embarked}.")

Imputed 'Embarked' with mode value: S.


/tmp/ipykernel_7173/2569350155.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Embarked'].fillna(mode_embarked, inplace=True)


### Re-evaluating Age Imputation: Group-Based Median

As discussed, using a group-based median for 'Age' imputation can be more effective than a single global median. We will group by 'Pclass' and 'Sex' to get more contextually relevant median ages.

In [ ]:
# To demonstrate the difference and implement group-based median imputation, I will re-load the original 'df' to ensure we start with the same missing 'Age' values.
# This step is important if you want to compare the effects; otherwise, you can skip re-loading if you just want to proceed with the improved imputation.
# For now, I will assume 'df' has not been reloaded and still contains the missing 'Age' values from the original state before the simple median imputation.

# Calculate median age grouped by 'Pclass' and 'Sex'
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

print("Imputed 'Age' using group-based median (by Pclass and Sex).")

Imputed 'Age' using group-based median (by Pclass and Sex).


In [ ]:
# Verify that all missing values, particularly in 'Age', have now been handled after this more sophisticated imputation.
print("\nMissing values after group-based age imputation:")
display(df.isnull().sum())


Missing values after group-based age imputation:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


## Feature Engineering

Let's create some new features that might be more predictive for survival.

In [ ]:
# Create 'FamilySize' feature
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

print("Created 'FamilySize' feature.")

# Display the first few rows with the new feature
display(df[['SibSp', 'Parch', 'FamilySize', 'Survived']].head())

Created 'FamilySize' feature.


,SibSp,Parch,FamilySize,Survived
0,1,0,2,0
1,1,0,2,1
2,0,0,1,1
3,1,0,2,1
4,0,0,1,0


In [ ]:
# Create 'IsAlone' feature
df['IsAlone'] = 0
df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

print("Created 'IsAlone' feature.")

# Display the first few rows with the new feature
display(df[['FamilySize', 'IsAlone', 'Survived']].head())

Created 'IsAlone' feature.


,FamilySize,IsAlone,Survived
0,2,0,0
1,2,0,1
2,1,1,1
3,2,0,1
4,1,1,0


### Encoding Categorical Features

To prepare the data for machine learning models, we need to convert categorical features into numerical representations. For the 'Sex' column, since it's a binary categorical variable ('male', 'female'), we can use label encoding.

In [ ]:
# Convert 'Sex' to numerical using label encoding (0 for female, 1 for male)
df['Sex'] = df['Sex'].map({'female': 0, 'male': 1}).astype(int)

print("Converted 'Sex' column to numerical.")

# Display the first few rows with the new 'Sex' encoding
display(df[['Sex', 'Survived']].head())

Converted 'Sex' column to numerical.


,Sex,Survived
0,1,0
1,0,1
2,0,1
3,0,1
4,1,0


### One-Hot Encoding for 'Embarked'

For categorical features with more than two unique values, like 'Embarked', one-hot encoding is a common approach. This creates new binary columns for each category, preventing the model from assuming any ordinal relationship between the categories.

In [ ]:
# Apply one-hot encoding to the 'Embarked' column
df = pd.get_dummies(df, columns=['Embarked'], prefix='Embarked', dtype=int)

print("Applied one-hot encoding to 'Embarked' column.")

# Display the first few rows with the new 'Embarked' encoded columns
display(df[['Embarked_C', 'Embarked_Q', 'Embarked_S', 'Survived']].head())

Applied one-hot encoding to 'Embarked' column.


,Embarked_C,Embarked_Q,Embarked_S,Survived
0,0,0,1,0
1,1,0,0,1
2,0,0,1,1
3,0,0,1,1
4,0,0,1,0


In [ ]:
# Verify the DataFrame information after encoding
print("\nDataFrame Info after encoding 'Embarked':")
df.info()


DataFrame Info after encoding 'Embarked':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    int64  
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  FamilySize   891 non-null    int64  
 11  IsAlone      891 non-null    int64  
 12  Embarked_C   891 non-null    int64  
 13  Embarked_Q   891 non-null    int64  
 14  Embarked_S   891 non-null    int64  
dtypes: float64(2), int64(11), object(2)
memory usage: 104.5+ KB


## Model Preparation: Feature Selection and Data Splitting

Before we can train any machine learning model, we need to clearly define our features (the independent variables that will be used to make predictions) and our target variable (what we are trying to predict). We also need to split our dataset into a training set and a testing set to evaluate the model's performance on unseen data.

### Feature Selection

Based on our data exploration and feature engineering, we will select the columns that we believe are most relevant for predicting 'Survived'. We'll drop columns that are identifiers ('PassengerId'), highly correlated with other features, or not useful for prediction ('Name', 'Ticket').

In [ ]:
# Drop irrelevant features
X = df.drop(['PassengerId', 'Name', 'Ticket', 'Survived'], axis=1)
y = df['Survived']

print("Selected features (X) and target variable (y).")
print("\nFeatures (X) head:")
display(X.head())
print("\nTarget (y) head:")
display(y.head())

Selected features (X) and target variable (y).

Features (X) head:


,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Embarked_C,Embarked_Q,Embarked_S
0,3,1,22.0,1,0,7.2500,2,0,0,0,1
1,1,0,38.0,1,0,71.2833,2,0,1,0,0
2,3,0,26.0,0,0,7.9250,1,1,0,0,1
3,1,0,35.0,1,0,53.1000,2,0,0,0,1
4,3,1,35.0,0,0,8.0500,1,1,0,0,1



Target (y) head:


,Survived
0,0
1,1
2,1
3,1
4,0


### Data Splitting

We will split the data into training and testing sets using `train_test_split` from `sklearn.model_selection`. This ensures that our model is evaluated on data it has not seen during training, providing a more reliable measure of its generalization performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

print("\nFirst 5 rows of X_train:")
display(X_train.head())

Training set size: 712 samples
Testing set size: 179 samples

First 5 rows of X_train:


,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Embarked_C,Embarked_Q,Embarked_S
331,1,1,45.5,0,0,28.5000,1,1,0,0,1
733,2,1,23.0,0,0,13.0000,1,1,0,0,1
382,3,1,32.0,0,0,7.9250,1,1,0,0,1
704,3,1,26.0,1,0,7.8542,2,0,0,0,1
813,3,0,6.0,4,2,31.2750,7,0,0,0,1


## Model Training: Logistic Regression

Logistic Regression is a linear model used for binary classification. Despite its name, it's a classification algorithm, not a regression algorithm. It works by estimating the probability that an instance belongs to a particular class (e.g., 'Survived' or 'Not Survived').

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the Logistic Regression model
model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets and handles L1/L2 regularization

# Train the model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


### Model Prediction and Evaluation

After training, we use the model to make predictions on the unseen test data (`X_test`) and then evaluate how well it performed against the actual outcomes (`y_test`).

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", class_report)


Accuracy: 0.7933

Confusion Matrix:
 [[90 15]
 [22 52]]

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.86      0.83       105
           1       0.78      0.70      0.74        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



## Next Steps: Feature Scaling and K-Nearest Neighbors (KNN)

Now that we have a baseline model, let's try another popular classification algorithm: **K-Nearest Neighbors (KNN)**. Before applying KNN, it's essential to perform **Feature Scaling**.

### Feature Scaling

Many machine learning algorithms, especially those that calculate distances between data points (like KNN), perform better when numerical input variables are scaled to a standard range. We'll use `StandardScaler` to transform our features so they have a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully using StandardScaler.")
print("\nFirst 5 rows of scaled X_train (numerical array):\n", X_train_scaled[:5])

Features scaled successfully using StandardScaler.

First 5 rows of scaled X_train (numerical array):
 [[-1.61413602  0.7243102   1.25364106 -0.47072241 -0.47934164 -0.07868358
  -0.55466613  0.81220297 -0.46146201 -0.30335547  0.59248936]
 [-0.40055118  0.7243102  -0.47728355 -0.47072241 -0.47934164 -0.37714494
  -0.55466613  0.81220297 -0.46146201 -0.30335547  0.59248936]
 [ 0.81303367  0.7243102   0.21508629 -0.47072241 -0.47934164 -0.47486697
  -0.55466613  0.81220297 -0.46146201 -0.30335547  0.59248936]
 [ 0.81303367  0.7243102  -0.24649361  0.37992316 -0.47934164 -0.47623026
   0.04009635 -1.23121934 -0.46146201 -0.30335547  0.59248936]
 [ 0.81303367 -1.38062393 -1.78509326  2.93185988  2.04874166 -0.02524937
   3.01390875 -1.23121934 -0.46146201 -0.30335547  0.59248936]]


### K-Nearest Neighbors (KNN)

KNN is a non-parametric, lazy learning algorithm. It classifies a data point based on how its neighbors are classified. The 'k' in KNN refers to the number of nearest neighbors considered. We will train a KNN model on our scaled data and evaluate its performance.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the KNN model
knn_model = KNeighborsClassifier(n_neighbors=5) # Starting with 5 neighbors as a common default

# Train the KNN model on the scaled training data
knn_model.fit(X_train_scaled, y_train)

print("K-Nearest Neighbors model trained successfully.")

K-Nearest Neighbors model trained successfully.


### KNN Model Prediction and Evaluation

Let's evaluate the performance of our KNN model on the scaled test data.

In [ ]:
# Make predictions on the scaled test set
y_pred_knn = knn_model.predict(X_test_scaled)

# Evaluate the KNN model
accuracy_knn = accuracy_score(y_test, y_pred_knn)
conf_matrix_knn = confusion_matrix(y_test, y_pred_knn)
class_report_knn = classification_report(y_test, y_pred_knn)

print(f"KNN Accuracy: {accuracy_knn:.4f}")
print("\nKNN Confusion Matrix:\n", conf_matrix_knn)
print("\nKNN Classification Report:\n", class_report_knn)

KNN Accuracy: 0.8045

KNN Confusion Matrix:
 [[90 15]
 [20 54]]

KNN Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.86      0.84       105
           1       0.78      0.73      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.80      0.80      0.80       179



In [ ]:
# Verify that all missing values have been handled
print("\nMissing values after imputation:")
display(df.isnull().sum())


Missing values after imputation:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [32]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest model
# Using a common number of estimators (trees) and a random state for reproducibility
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the Random Forest model on the scaled training data
rf_model.fit(X_train_scaled, y_train)

print("Random Forest Classifier model trained successfully.")

# Make predictions on the scaled test set
y_pred_rf = rf_model.predict(X_test_scaled)

# Evaluate the Random Forest model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)
class_report_rf = classification_report(y_test, y_pred_rf)

print(f"\nRandom Forest Accuracy: {accuracy_rf:.4f}")
print("\nRandom Forest Confusion Matrix:\n", conf_matrix_rf)
print("\nRandom Forest Classification Report:\n", class_report_rf)

Random Forest Classifier model trained successfully.

Random Forest Accuracy: 0.8268

Random Forest Confusion Matrix:
 [[91 14]
 [17 57]]

Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.87      0.85       105
           1       0.80      0.77      0.79        74

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



## Exploring More Models

### Decision Tree Classifier

Let's start by implementing a Decision Tree Classifier. Decision Trees are non-parametric supervised learning methods used for classification and regression. The goal is to create a model that predicts the value of a target variable by learning simple decision rules inferred from the data features.

In [35]:
from sklearn.tree import DecisionTreeClassifier

# Initialize the Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42)

# Train the Decision Tree model on the scaled training data
dt_model.fit(X_train_scaled, y_train)

print("Decision Tree Classifier model trained successfully.")

# Make predictions on the scaled test set
y_pred_dt = dt_model.predict(X_test_scaled)

# Evaluate the Decision Tree model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
conf_matrix_dt = confusion_matrix(y_test, y_pred_dt)
class_report_dt = classification_report(y_test, y_pred_dt)

print(f"\nDecision Tree Accuracy: {accuracy_dt:.4f}")
print("\nDecision Tree Confusion Matrix:\n", conf_matrix_dt)
print("\nDecision Tree Classification Report:\n", class_report_dt)

Decision Tree Classifier model trained successfully.

Decision Tree Accuracy: 0.7709

Decision Tree Confusion Matrix:
 [[84 21]
 [20 54]]

Decision Tree Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.80      0.80       105
           1       0.72      0.73      0.72        74

    accuracy                           0.77       179
   macro avg       0.76      0.76      0.76       179
weighted avg       0.77      0.77      0.77       179



### Support Vector Machine (SVM)

Next, we'll try a Support Vector Machine (SVM) with a radial basis function (RBF) kernel. SVMs are powerful and versatile machine learning models capable of performing linear or non-linear classification, regression, and even outlier detection. They work by finding the hyperplane that best separates the classes in the feature space.

In [34]:
from sklearn.svm import SVC

# Initialize the SVM model with a radial basis function (RBF) kernel
svm_model = SVC(kernel='rbf', random_state=42)

# Train the SVM model on the scaled training data
svm_model.fit(X_train_scaled, y_train)

print("Support Vector Machine model trained successfully.")

# Make predictions on the scaled test set
y_pred_svm = svm_model.predict(X_test_scaled)

# Evaluate the SVM model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
conf_matrix_svm = confusion_matrix(y_test, y_pred_svm)
class_report_svm = classification_report(y_test, y_pred_svm)

print(f"\nSVM Accuracy: {accuracy_svm:.4f}")
print("\nSVM Confusion Matrix:\n", conf_matrix_svm)
print("\nSVM Classification Report:\n", class_report_svm)

Support Vector Machine model trained successfully.

SVM Accuracy: 0.8156

SVM Confusion Matrix:
 [[93 12]
 [21 53]]

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.89      0.85       105
           1       0.82      0.72      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179



### Gradient Boosting Classifier

Finally, let's implement a Gradient Boosting Classifier. Gradient Boosting is a powerful ensemble technique that builds models sequentially, with each new model correcting errors made by previous ones. It is known for its high predictive accuracy.

In [33]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize the Gradient Boosting model
gbc_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Train the Gradient Boosting model on the scaled training data
gbc_model.fit(X_train_scaled, y_train)

print("Gradient Boosting Classifier model trained successfully.")

# Make predictions on the scaled test set
y_pred_gbc = gbc_model.predict(X_test_scaled)

# Evaluate the Gradient Boosting model
accuracy_gbc = accuracy_score(y_test, y_pred_gbc)
conf_matrix_gbc = confusion_matrix(y_test, y_pred_gbc)
class_report_gbc = classification_report(y_test, y_pred_gbc)

print(f"\nGradient Boosting Accuracy: {accuracy_gbc:.4f}")
print("\nGradient Boosting Confusion Matrix:\n", conf_matrix_gbc)
print("\nGradient Boosting Classification Report:\n", class_report_gbc)

Gradient Boosting Classifier model trained successfully.

Gradient Boosting Accuracy: 0.8045

Gradient Boosting Confusion Matrix:
 [[92 13]
 [22 52]]

Gradient Boosting Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84       105
           1       0.80      0.70      0.75        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



## Hyperparameter Tuning for Random Forest Classifier

Given that Random Forest showed promising results, let's try to optimize its performance further through hyperparameter tuning. We will use `GridSearchCV` to explore a range of hyperparameter values and find the combination that yields the best performance on our dataset.

In [37]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200],  # Number of trees in the forest
    'max_features': ['sqrt', 'log2'],  # Number of features to consider for each split
    'max_depth': [None, 10, 20, 30], # Maximum depth of the tree
    'min_samples_split': [2, 5, 10], # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]   # Minimum number of samples required to be at a leaf node
}

# Initialize the Random Forest Classifier
rf_base_model = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV
# 'cv=5' means 5-fold cross-validation
# 'scoring="accuracy"' specifies the metric to optimize
# 'n_jobs=-1' uses all available CPU cores
grid_search_rf = GridSearchCV(estimator=rf_base_model, param_grid=param_grid_rf,
                              cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

print("Starting GridSearchCV for Random Forest. This may take some time...")

# Fit GridSearchCV to the scaled training data
grid_search_rf.fit(X_train_scaled, y_train)

print("GridSearchCV completed.")

Starting GridSearchCV for Random Forest. This may take some time...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
GridSearchCV completed.


### Best Hyperparameters and Model Evaluation

After `GridSearchCV` completes, we can inspect the best parameters found and evaluate the performance of the Random Forest model with these optimized settings.

In [38]:
# Get the best parameters found by GridSearchCV
best_params_rf = grid_search_rf.best_params_
print(f"Best Random Forest Hyperparameters: {best_params_rf}")

# Get the best model from GridSearchCV
best_rf_model = grid_search_rf.best_estimator_

# Make predictions with the best model on the scaled test set
y_pred_best_rf = best_rf_model.predict(X_test_scaled)

# Evaluate the best Random Forest model
accuracy_best_rf = accuracy_score(y_test, y_pred_best_rf)
conf_matrix_best_rf = confusion_matrix(y_test, y_pred_best_rf)
class_report_best_rf = classification_report(y_test, y_pred_best_rf)

print(f"\nOptimized Random Forest Accuracy: {accuracy_best_rf:.4f}")
print("\nOptimized Random Forest Confusion Matrix:\n", conf_matrix_best_rf)
print("\nOptimized Random Forest Classification Report:\n", class_report_best_rf)

Best Random Forest Hyperparameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}

Optimized Random Forest Accuracy: 0.8268

Optimized Random Forest Confusion Matrix:
 [[96  9]
 [22 52]]

Optimized Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.91      0.86       105
           1       0.85      0.70      0.77        74

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.82       179



## Summary of Model Performances

Let's compile the accuracy scores from all the models we've trained and evaluated to compare their performance side-by-side.

In [39]:
model_accuracies = {
    'Logistic Regression': accuracy,
    'K-Nearest Neighbors': accuracy_knn,
    'Random Forest (Initial)': accuracy_rf,
    'Random Forest (Optimized)': accuracy_best_rf,
    'Decision Tree': accuracy_dt,
    'Support Vector Machine': accuracy_svm,
    'Gradient Boosting': accuracy_gbc
}

# Create a DataFrame to display the accuracies
accuracy_df = pd.DataFrame(model_accuracies.items(), columns=['Model', 'Accuracy'])
accuracy_df = accuracy_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

print("\nModel Comparison:")
display(accuracy_df)


Model Comparison:


,Model,Accuracy
0,Random Forest (Initial),0.826816
1,Random Forest (Optimized),0.826816
2,Support Vector Machine,0.815642
3,Gradient Boosting,0.804469
4,K-Nearest Neighbors,0.804469
5,Logistic Regression,0.793296
6,Decision Tree,0.770950


The comparison table shows the accuracy of each model. While the optimized Random Forest is among the best, the improvement from the initial Random Forest is marginal.

Here are some potential next steps we could consider:

1.  **More Extensive Hyperparameter Tuning**: We could expand the `param_grid` for Random Forest or other promising models, or use `RandomizedSearchCV` for a more efficient search over larger parameter spaces.
2.  **Ensemble Methods**: Explore advanced ensemble techniques like Bagging, AdaBoost, or Stacked models, which combine multiple models to often achieve higher performance than individual models.
3.  **Feature Engineering**: Go deeper into creating new, more informative features from existing ones. For example, extracting titles from 'Name', or categorizing 'Fare' into bins.
4.  **Cross-Validation**: Implement more robust cross-validation (e.g., K-Fold Cross-Validation) during model evaluation, not just during hyperparameter tuning, to get a more reliable estimate of model performance.
5.  **Error Analysis**: Analyze the predictions of our best model (Random Forest) to understand where it's making mistakes. Are there specific types of passengers it consistently misclassifies? This can lead to insights for new features or data cleaning.
6.  **Addressing Class Imbalance**: Although not explicitly observed as a major issue, if one class significantly outnumbers the other, techniques like SMOTE (Synthetic Minority Over-sampling Technique) could be considered.

Which of these options would you like to explore next?